In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import re

raw_path = "/Volumes/workspace/default/census360_raw/"

gn_population_file = raw_path + "GN_population_excel.xlsx"
gn_housing_file = raw_path + "GN_housing_excel.xlsx"
trcsl_file = raw_path + "TRCSL_Telecom_Statistics_Q1_2026.xlsx"
gnd_list_file = raw_path + "GNDList_Final.xlsx"
district_population_file = raw_path + "population_by_district_in_census_years.csv"
population_2012_file = raw_path + "population_2012_2.csv"

target_schema = "workspace.census360"

print("Cell 1 completed: paths are ready")

In [0]:
def clean_column_name(col):
    col = str(col)
    col = col.strip()
    col = col.replace("\n", " ")
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    col = col.lower()
    return col


def clean_pandas_df(df):
    df = df.copy()
    
    # Clean column names
    df.columns = [clean_column_name(c) for c in df.columns]
    
    # Replace NaN with None
    df = df.replace({np.nan: None})
    
    return df


def pandas_to_spark_safe(df):
    df = clean_pandas_df(df)
    
    # Convert through records to avoid Arrow conversion issues
    records = df.to_dict(orient="records")
    
    return spark.createDataFrame(records)


print("Cell 2 completed: helper functions are ready")

In [0]:
gn_pop_raw = pd.read_excel(
    gn_population_file,
    sheet_name="Population",
    header=None,
    engine="openpyxl"
)

gn_pop = gn_pop_raw.iloc[5:].copy()

gn_pop.columns = [
    "province_code",
    "province_name",
    "district_code",
    "district_name",
    "ds_division_code",
    "ds_division_name",
    "gn_division_code",
    "gn_division_name",
    "gn_division_number",
    "population_total",
    "male_population",
    "female_population",
    "age_total",
    "age_0_14",
    "age_15_59",
    "age_60_64",
    "age_65_plus"
]

gn_pop = gn_pop.dropna(how="all")

print("Cell 3 completed: GN Population loaded")
print("Shape:", gn_pop.shape)
print(gn_pop.head(5).to_string())

In [0]:
gn_housing = pd.read_excel(
    gn_housing_file,
    sheet_name="Housing Units",
    header=3,
    engine="openpyxl"
)

gn_housing = gn_housing.dropna(how="all")

gn_housing.columns = [
    "province_code",
    "province_name",
    "district_code",
    "district_name",
    "ds_division_code",
    "ds_division_name",
    "gn_division_code",
    "gn_division_name",
    "gn_division_number",
    "occupied_housing_units"
]

print("Cell 4 completed: GN Housing loaded")
print("Shape:", gn_housing.shape)
print(gn_housing.head(5).to_string())

In [0]:
gnd_list = pd.read_excel(
    gnd_list_file,
    sheet_name="GNDList",
    header=0,
    engine="openpyxl"
)

gnd_list = gnd_list.dropna(how="all")

print("Cell 5 completed: GND List loaded")
print("Shape:", gnd_list.shape)
print(gnd_list.head(5).to_string())
print("Columns:", gnd_list.columns.tolist())

In [0]:
trcsl = pd.read_excel(
    trcsl_file,
    sheet_name="Project Dataset",
    header=2,
    engine="openpyxl"
)

trcsl = trcsl.dropna(how="all")

print("Cell 6 completed: TRCSL Project Dataset loaded")
print("Shape:", trcsl.shape)
print(trcsl.head(5).to_string())
print("Columns:", trcsl.columns.tolist())

In [0]:
district_population = pd.read_csv(district_population_file)
population_2012 = pd.read_csv(population_2012_file)

print("Cell 7 completed: CSV files loaded")

print("District population shape:", district_population.shape)
print(district_population.head(5).to_string())
print("District population columns:", district_population.columns.tolist())

print("=" * 100)

print("Population 2012 shape:", population_2012.shape)
print(population_2012.head(5).to_string())
print("Population 2012 columns:", population_2012.columns.tolist())

In [0]:
district_population = clean_pandas_df(district_population)
population_2012 = clean_pandas_df(population_2012)

print("Cell 8 completed: CSV column names cleaned")

print("District population columns:")
print(district_population.columns.tolist())

print("=" * 100)

print("Population 2012 columns:")
print(population_2012.columns.tolist())

In [0]:
bronze_gn_population_sdf = pandas_to_spark_safe(gn_pop)
bronze_gn_housing_sdf = pandas_to_spark_safe(gn_housing)
bronze_gnd_list_sdf = pandas_to_spark_safe(gnd_list)
bronze_trcsl_sdf = pandas_to_spark_safe(trcsl)
bronze_district_population_sdf = pandas_to_spark_safe(district_population)
bronze_population_2012_sdf = pandas_to_spark_safe(population_2012)

print("Cell 9 completed: all pandas DataFrames converted to Spark")

In [0]:
print("GN Population")
bronze_gn_population_sdf.show(5, truncate=False)

print("GN Housing")
bronze_gn_housing_sdf.show(5, truncate=False)

print("GND List")
bronze_gnd_list_sdf.show(5, truncate=False)

print("TRCSL")
bronze_trcsl_sdf.show(5, truncate=False)

print("District Population")
bronze_district_population_sdf.show(5, truncate=False)

print("Population 2012")
bronze_population_2012_sdf.show(5, truncate=False)

print("Cell 10 completed: Spark previews displayed")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.census360")

print("Cell 11 completed: schema workspace.census360 is ready")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.census360.bronze_gn_population")

print("Old bronze_gn_population table dropped")

In [0]:
bronze_gn_population_sdf.write.mode("overwrite").saveAsTable("workspace.census360.bronze_gn_population")

print("Saved: bronze_gn_population")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.census360.bronze_gn_housing")

bronze_gn_housing_sdf.write.mode("overwrite").saveAsTable("workspace.census360.bronze_gn_housing")

print("Saved: bronze_gn_housing")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.census360.bronze_gnd_list")

bronze_gnd_list_sdf.write.mode("overwrite").saveAsTable("workspace.census360.bronze_gnd_list")

print("Saved: bronze_gnd_list")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.census360.bronze_trcsl_telecom_statistics")

bronze_trcsl_sdf.write.mode("overwrite").saveAsTable("workspace.census360.bronze_trcsl_telecom_statistics")

print("Saved: bronze_trcsl_telecom_statistics")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.census360.bronze_population_by_district_census_years")

bronze_district_population_sdf.write.mode("overwrite").saveAsTable(
    "workspace.census360.bronze_population_by_district_census_years"
)

print("Saved: bronze_population_by_district_census_years")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.census360.bronze_population_2012")

bronze_population_2012_sdf.write.mode("overwrite").saveAsTable(
    "workspace.census360.bronze_population_2012"
)

print("Saved: bronze_population_2012")

In [0]:
spark.sql("SHOW TABLES IN workspace.census360").show(truncate=False)

In [0]:
bronze_tables = [
    "bronze_gn_population",
    "bronze_gn_housing",
    "bronze_gnd_list",
    "bronze_trcsl_telecom_statistics",
    "bronze_population_by_district_census_years",
    "bronze_population_2012"
]

for table in bronze_tables:
    count = spark.table(f"workspace.census360.{table}").count()
    print(f"{table}: {count}")